# Ticket 3 - Indexacion vectorial en Colab

Este notebook funciona como laboratorio y evidencia del Ticket 3 del proyecto **BlueSea AI Assistant**.

Objetivo del notebook:

- instalar el proyecto en Google Colab;
- confirmar que exista `data/processed/chunks.jsonl` como salida del Ticket 2;
- generar embeddings locales deterministicos para cada chunk;
- crear el indice local `data/index/vectors.jsonl`;
- generar el manifest tecnico `data/index/embeddings_manifest.json`;
- probar una busqueda tecnica con `search-index`.

El notebook no reemplaza el codigo final. La logica principal vive en `src/rag_bsf/` y este cuaderno solo la ejecuta de forma demostrativa.

## 1. Clonar el repositorio

Ejecuta esta celda en Google Colab. Si el repositorio ya existe en la sesion, puedes reiniciar el runtime o borrar la carpeta antes de clonar otra vez.

In [ ]:
!git clone https://github.com/aantoa/bluesea-ai-assistant.git
%cd bluesea-ai-assistant

## 2. Instalar dependencias y paquete local

`requirements.txt` incluye `-e .`, por eso Colab instala el paquete desde `src/rag_bsf/` y permite ejecutar `python -m rag_bsf.cli`.

In [ ]:
!python --version
!pip install -r requirements.txt

## 3. Confirmar salida del Ticket 2

El Ticket 3 no procesa documentos completos directamente. Su entrada principal es `data/processed/chunks.jsonl`, generado por el Ticket 2.

Si el archivo no existe o esta vacio, puedes ejecutar primero `python -m rag_bsf.cli process`. En Colab, si no subiste documentos Markdown locales, es normal obtener `0 chunks`.

In [4]:
from pathlib import Path

# Buscar la raíz del proyecto aunque estés dentro de /notebooks
current = Path.cwd()
project_root = current

while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

chunks_path = project_root / "data" / "processed" / "chunks.jsonl"

if chunks_path.exists():
    print("chunks.jsonl existe")
    print("Tamanio bytes:", chunks_path.stat().st_size)
    print("Lineas:", sum(1 for _ in chunks_path.open(encoding="utf-8")))
else:
    print("chunks.jsonl no existe todavia")

chunks.jsonl existe
Tamanio bytes: 181567
Lineas: 269


## 4. Generar chunks si hace falta

Esta celda es opcional, pero ayuda a dejar el flujo completo en Colab: valida que el Ticket 2 produzca `chunks.jsonl` antes de indexar.

Si no hay documentos Markdown locales en `documents/<area>/`, el resultado puede ser `0 documents, 0 chunks`.

In [ ]:
!python -m rag_bsf.cli process

## 5. Ejecutar indexacion vectorial

Este comando lee `data/processed/chunks.jsonl`, genera embeddings locales con `HashingEmbedder` y guarda el indice en `data/index/`.

In [ ]:
!python -m rag_bsf.cli index

## 6. Revisar archivos generados

Las salidas principales del Ticket 3 son locales y no deben subirse a GitHub.

In [ ]:
!find data/index -maxdepth 1 -type f -print 2>/dev/null | sort || true
!wc -l data/index/vectors.jsonl 2>/dev/null || true
!ls -lh data/index 2>/dev/null || true

## 7. Inspeccionar manifest tecnico

El manifest registra trazabilidad de la indexacion: archivo fuente, hash, modelo local, dimensiones y cantidad de vectores.

In [3]:
import json
from pathlib import Path

# Buscar la raíz del proyecto aunque estés dentro de /notebooks
current = Path.cwd()
project_root = current

while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

manifest_path = project_root / "data" / "index" / "embeddings_manifest.json"

if not manifest_path.exists():
    print("No existe embeddings_manifest.json. Ejecuta primero el comando index.")
else:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "generated_at": "2026-07-21T13:22:42.527961+00:00",
  "source_chunks_file": "/Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/processed/chunks.jsonl",
  "source_chunks_sha256": "35ae03e56120966b51d3d7a1ea705f79c824cedec3be8b2daa261d1e5a916924",
  "vector_index_file": "/Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/index/vectors.jsonl",
  "embedding_model": "local-hashing-v1",
  "dimensions": 384,
  "chunks": 269,
  "vectors": 269
}


## 8. Inspeccionar vectores

Cada linea de `vectors.jsonl` guarda un chunk, su vector numerico y la metadata necesaria para recuperar la fuente.

In [2]:
import json
from pathlib import Path

# Buscar la raíz del proyecto aunque estés dentro de /notebooks
current = Path.cwd()
project_root = current

while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

vectors_path = project_root / "data" / "index" / "vectors.jsonl"

if not vectors_path.exists() or vectors_path.stat().st_size == 0:
    print("No hay vectores disponibles todavia. Revisa que existan chunks y vuelve a ejecutar index.")
else:
    with vectors_path.open(encoding="utf-8") as fh:
        for index, line in zip(range(3), fh):
            item = json.loads(line)
            chunk = item.get("chunk", {})
            metadata = chunk.get("metadata", {})
            vector = item.get("vector", [])

            print(f"Registro vectorial: {index + 1}")
            print(f"Modelo: {item.get('embedding_model')} | Dimensiones: {item.get('dimensions')}")
            print(f"Chunk: {chunk.get('chunk_id')}")
            print(f"Documento: {metadata.get('document_code')} - {metadata.get('title')}")
            print(f"Seccion: {metadata.get('section')}")
            print("Primeros 12 valores del vector:", vector[:12])
            print("Texto:", chunk.get("text", "")[:500])
            print("-" * 100)

Registro vectorial: 1
Modelo: local-hashing-v1 | Dimensiones: 384
Chunk: BSF-HR-002::chunk-0001
Documento: BSF-HR-002 - Employee FAQ
Seccion: BlueSea Foods
Primeros 12 valores del vector: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Texto: # BlueSea Foods
----------------------------------------------------------------------------------------------------
Registro vectorial: 2
Modelo: local-hashing-v1 | Dimensiones: 384
Chunk: BSF-HR-002::chunk-0002
Documento: BSF-HR-002 - Employee FAQ
Seccion: Employee FAQ
Primeros 12 valores del vector: [0.0, 0.0, -0.08638684255813601, 0.0, 0.0, 0.08638684255813601, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Texto: # Employee FAQ

Document ID: BSF-HR-002 
Department: Human Resources 
Owner: Human Resources Manager 
Approver: General Manager 
Version: 1.0 
Status: Active 
Effective Date: July 1, 2026 
Review Date: July 1, 2027 
Confidentiality: Internal 
Language: English 
Keywords: employee FAQ, human resources, payroll, benefits, leave, attendance

## 9. Probar busqueda tecnica

`search-index` no genera una respuesta final. Solo devuelve los chunks mas cercanos al texto de consulta para validar que el indice local puede recuperarse.

In [ ]:
!python -m rag_bsf.cli search-index "politica de confidencialidad y manejo documental" --top-k 5

## 10. Criterio para cerrar Ticket 3

Antes de avanzar al siguiente ticket, revisa que:

- `data/processed/chunks.jsonl` exista como salida del Ticket 2;
- `data/index/vectors.jsonl` se genere sin errores;
- `data/index/embeddings_manifest.json` registre modelo, dimensiones, chunks y vectores;
- `search-index` permita inspeccionar chunks cercanos cuando existan vectores;
- el notebook se use como evidencia, pero el codigo final se mantenga en `src/rag_bsf/`.

El Ticket 3 no implementa todavia agente conversacional, generacion de respuestas ni interfaz.